# Swiss Citation — Precision v1  (variable-K from snapshot)

A separate, precision-driven pipeline that reads the v7.5 snapshot (top-50,000 per query, ~0.90 macro recall) and compresses it to a per-query gold-citation set of *variable size* — driven by LLM conviction + query-aspect coverage, NOT a fixed top-K.

## Architecture (5 narrowing stages)

```
50k per query   (from snapshot — R@50k ≈ 0.901)
  │
  ▼  Stage A — cheap rule filter (no LLM)
     drop: cantonal court · facts/procedural/notification/costs roles
           · dispositif-regex hits · zero-signal
  ▼  ~5k-20k per query
  │
  ▼  Stage B — Phase-1 dossier composite score (no LLM)
     7 features: lead-statute ∩ · full-statute ∩ · co-citation density
                 · concept ∩ · aspect best-match · chamber × area
                 · paragraph_role · doctrinal density
  ▼  top-500 per query
  │
  ▼  Stage C — listwise LLM with dossier (Qwen3-32B, 15/batch)
     1-5 score + addresses_aspect + ≤20-word why; keep score ≥ 3
  ▼  ~100-200 per query
  │
  ▼  Stage D — pointwise judgment with EVIDENCE QUOTE (Qwen3-32B, 5/batch)
     verbatim-substring requirement is the anti-hallucination gate;
     verdict deterministically computed from (confidence, quote)
  ▼  KEEP / MAYBE / DROP
  │
  ▼  Stage E — adaptive selection (no LLM)
     all KEEP@5 → all KEEP@4 → fill aspect gaps from MAYBE → case-level dedup
  ▼
  ▼  Stage F — self-validation pass (Qwen3-32B, 1 call per query)
     "Is each finalist necessary or redundant given the others?"
  ▼
  VARIABLE-K final set per query
```

## Anti-hallucination invariants (enforced, not requested)

- Stage B: must have ≥1 statute / concept / term hit OR be co-cited
- Stage D: evidence_quote must be a VERBATIM substring of doc text (regex-checked)
- Stage D: verdict computed from confidence + quote, not trusted from LLM output
- Stage E: case-level dedup (no double-counting same case)
- Stage F: never drop the only candidate for an unaddressed aspect

## Prerequisites

Run the v7.5 multi-query notebook through **cell 48 with the top-50,000 patch** (cell 48 source must say `[:50000]` not `[:5000]`). That produces the snapshot at:
```
/content/drive/MyDrive/swiss_law/research/anchor_funnel_val001_v7/snapshot/
```
This precision-v1 notebook reads it in ~60s and runs the 5-stage cascade.

## Output

`OUT_DIR/precision_v1_summary.json` — full state + per-query P/R/F1 + comparison vs fusion baseline.
`OUT_DIR/precision_v1_predictions.json` — citation strings per query (Kaggle-style).


# Phase 0 — Setup (imports + path resolution)


In [1]:
# Phase 0 — Setup: imports + path resolution + vLLM/FlashInfer install
# (Blackwell-optimized — pattern from enrich_laws_de_qwen3_8b_kaggle reference notebook)

import sys, os, json, gzip, time, re, gc, subprocess
import importlib.util
from pathlib import Path
from collections import defaultdict, Counter

print(f"Python {sys.version_info.major}.{sys.version_info.minor}")

# Environment detection
try:
    from google.colab import drive as _drv
    _drv.mount("/content/drive", force_remount=False)
    _DRIVE = Path("/content/drive/MyDrive/swiss_law")
    ENV = "colab"
except (ImportError, ModuleNotFoundError):
    _DRIVE = Path(r"E:\swiss_citation_extraction")
    ENV = "local"
IN_KAGGLE = bool(os.environ.get("KAGGLE_URL_BASE") or Path("/kaggle").exists())
print(f"Environment: {ENV}{' (Kaggle)' if IN_KAGGLE else ''}")

SNAPSHOT_DIR = _DRIVE / "research" / "anchor_funnel_val001_v7" / "snapshot"
OUT_DIR      = _DRIVE / "research" / "precision_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert SNAPSHOT_DIR.exists(), (
    f"Snapshot dir missing: {SNAPSHOT_DIR}\n"
    f"Run cell 48 of swiss_citation_anchor_funnel_v7_5_multiquery (12).ipynb first."
)
print(f"snapshot: {SNAPSHOT_DIR}")
print(f"out:      {OUT_DIR}")

# ─── vLLM + FlashInfer install (Blackwell-optimized one-time install) ──────
_NEEDS_RESTART = False

if importlib.util.find_spec("vllm") is None:
    print("\n[setup] installing core packages (~3 min)...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "vllm>=0.10.0", "transformers>=4.51.0",
        "accelerate", "safetensors", "huggingface_hub",
    ], check=True)
    _NEEDS_RESTART = True

if importlib.util.find_spec("flashinfer") is None:
    print("[setup] installing FlashInfer (CUDA-version-aware)...")
    def _cuda_suffix():
        try:
            import torch as _t
            cuda = (_t.version.cuda or "").strip()
            if not cuda: return None
            parts = cuda.split(".")
            major, minor = int(parts[0]), int(parts[1])
            supported = {"cu126", "cu128", "cu129", "cu130", "cu131"}
            s = f"cu{major}{minor}"
            if s in supported: return s
            cands = sorted(supported, key=lambda x: int(x[2:]))
            det = major * 10 + minor
            best = None
            for c in cands:
                if int(c[2:]) <= det: best = c
            return best or cands[0]
        except Exception:
            return None
    _cidx = _cuda_suffix()
    print(f"  CUDA index suffix: {_cidx}")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                        "flashinfer-python", "flashinfer-cubin"], check=False)
        if _cidx:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "--pre",
                            "flashinfer-jit-cache",
                            "--index-url", f"https://flashinfer.ai/whl/{_cidx}"], check=False)
        os.environ["FLASHINFER_DISABLE_VERSION_CHECK"] = "1"
        importlib.invalidate_caches()
        import flashinfer
        print(f"  flashinfer {getattr(flashinfer, '__version__', '?')} OK")
        _NEEDS_RESTART = True
    except Exception as e:
        print(f"  flashinfer install skipped ({e}); vLLM auto-selects backend (still fast).")

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

if _NEEDS_RESTART:
    print("\n" + "=" * 64)
    print("IMPORTANT: this was a FIRST install. RESTART THE RUNTIME ONCE")
    print("(Runtime → Restart runtime in Colab; Kernel → Restart in Jupyter),")
    print("then re-run this cell. The runtime restart is needed for vLLM's")
    print("and FlashInfer's CUDA extensions to bind cleanly. Cell 4 onwards")
    print("will fail without the restart.")
    print("=" * 64)
    raise SystemExit("Restart runtime and re-run from this cell.")
else:
    print("\n[setup] vLLM + FlashInfer already installed; proceeding.")


Python 3.12
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment: colab (Kaggle)
snapshot: /content/drive/MyDrive/swiss_law/research/anchor_funnel_val001_v7/snapshot
out:      /content/drive/MyDrive/swiss_law/research/precision_v1

[setup] vLLM + FlashInfer already installed; proceeding.


In [2]:
import flashinfer
print(f"  flashinfer {getattr(flashinfer, '__version__', '?')} OK")

  flashinfer 0.6.8.post1 OK


# Phase 1 — Warm-Boot from snapshot


In [3]:
# Phase 1 — Warm-Boot from snapshot
# Reconstructs: CONFIG, ALL_QUERIES, search_text, doc_meta, doc_statute_anchors,
# _doc_to_concepts, _doc_to_terms, PER_QUERY, ALL_TARGETS, ALL_HYDE_ASPECTS,
# ALL_GOLD_DOC_SET, ALL_TOTAL_GOLD from snapshot in ~30-60s.
#
# Pool ceiling: snapshot has top-5000 per query. Macro mean R@5000 = 0.728.
# That's our recall headroom — the cascade compresses 5000 → variable-K gold.

import json as _j, gzip as _gz
import pandas as _pd

print(f"[warm-boot] loading from {SNAPSHOT_DIR}")
_t0 = time.time()

# CONFIG
CONFIG = _j.loads((SNAPSHOT_DIR / "config.json").read_text(encoding="utf-8"))
print(f"  CONFIG loaded (topk_final={CONFIG.get('topk_final')})")

# Resolve val.csv — paths.json may reference Drive; fall back to local
_paths_meta = _j.loads((SNAPSHOT_DIR / "paths.json").read_text(encoding="utf-8"))
VAL_CSV = Path(_paths_meta["val_csv"])
if not VAL_CSV.exists():
    VAL_CSV = _DRIVE / "data" / "val.csv"
assert VAL_CSV.exists(), f"val.csv not found at {VAL_CSV}"
val_df = _pd.read_csv(VAL_CSV)
ALL_QUERIES = [
    {"query_id":   str(r["query_id"]),
     "query_text": str(r["query"]),
     "gold":       [c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()]}
    for _, r in val_df.iterrows()
]
ALL_TOTAL_GOLD = {q["query_id"]: len(q["gold"]) for q in ALL_QUERIES}
print(f"  ALL_QUERIES: {len(ALL_QUERIES)}  ({VAL_CSV.name})")

# Corpus snapshot — compressed JSON, ~15 MB → unpacks to ~80-150 MB in memory
with _gz.open(SNAPSHOT_DIR / "corpus_snapshot.json.gz", "rt", encoding="utf-8") as _f:
    _c = _j.load(_f)
search_text         = {_d: _r["ct"]  for _d, _r in _c.items()}
doc_meta            = {_d: {"citation":       _r["cit"],
                            "family":         _r["fam"],
                            "court_base":     _r["cb"],
                            "paragraph_role": _r["pr"],
                            "language":       _r["ln"]}
                       for _d, _r in _c.items()}
doc_statute_anchors = {_d: set(_r["sa"]) for _d, _r in _c.items()}
_doc_to_concepts    = {_d: set(_r["cn"]) for _d, _r in _c.items()}
_doc_to_terms       = {_d: set(_r["tm"]) for _d, _r in _c.items()}
print(f"  corpus_snapshot: {len(_c):,} unique docs (cascade pool)")

# PER_QUERY — final_topk[:5000] per query
_pq = _j.loads((SNAPSHOT_DIR / "per_query_snapshot.json").read_text(encoding="utf-8"))
PER_QUERY = {
    _qid: {
        "final_topk":      _r["final_topk"],
        "curve":           {int(_k): tuple(_v) for _k, _v in _r.get("curve", {}).items()},
        "channel_recalls": {_ch: tuple(_v) for _ch, _v in _r.get("channel_recalls", {}).items()},
        "gold":            _r.get("gold", 0),
        "gold_doc_ids":    _r.get("gold_doc_ids", 0),
        "R_at_K":          _r.get("R_at_K", 0.0),
    }
    for _qid, _r in _pq.items()
}
print(f"  PER_QUERY: {len(PER_QUERY)}")

# Targets, aspects, gold
ALL_TARGETS = (_j.loads((SNAPSHOT_DIR / "all_targets.json").read_text(encoding="utf-8"))
               if (SNAPSHOT_DIR / "all_targets.json").exists() else {})
ALL_HYDE_ASPECTS = (_j.loads((SNAPSHOT_DIR / "hyde_aspects.json").read_text(encoding="utf-8"))
                    if (SNAPSHOT_DIR / "hyde_aspects.json").exists() else {})
_g = _j.loads((SNAPSHOT_DIR / "gold_doc_sets.json").read_text(encoding="utf-8"))
ALL_GOLD_DOC_SET = {_qid: set(_lst) for _qid, _lst in _g.items()}
print(f"  ALL_TARGETS: {len(ALL_TARGETS)}")
print(f"  ALL_HYDE_ASPECTS: {len(ALL_HYDE_ASPECTS)}")
print(f"  ALL_GOLD_DOC_SET total: {sum(len(_s) for _s in ALL_GOLD_DOC_SET.values()):,} doc_ids")

# Quick sanity: the first query's top-2 docs
_sqid = ALL_QUERIES[0]["query_id"]
_sample = PER_QUERY[_sqid]["final_topk"][:2]
print(f"\n  sanity check ({_sqid}, top-2 = {_sample}):")
for _d in _sample:
    _txt = (search_text.get(_d, "") or "")[:140].replace("\n", " ")
    print(f"    {_d}: {_txt!r}")

print(f"\n[warm-boot] done in {time.time()-_t0:.1f}s")


[warm-boot] loading from /content/drive/MyDrive/swiss_law/research/anchor_funnel_val001_v7/snapshot
  CONFIG loaded (topk_final=50000)
  ALL_QUERIES: 10  (val.csv)
  corpus_snapshot: 255,713 unique docs (cascade pool)
  PER_QUERY: 10
  ALL_TARGETS: 10
  ALL_HYDE_ASPECTS: 10
  ALL_GOLD_DOC_SET total: 254 doc_ids

  sanity check (val_001, top-2 = ['law:128343', 'law:57057']):
    law:128343: 'Art. 10 Abs. 1 StGB This article distinguishes between crimes and offenses based on the severity of penalties threatened. criminal acts pena'
    law:57057: 'Art. 66 Abs. 1 BGG The article states that court costs are generally imposed on the losing party, but the Federal Court may distribute them '

[warm-boot] done in 5.6s


# Phase 2 — Phase-1 Dossier

## 2a. Helpers (chamber regex, doctrinal density, legal-area inference)


In [4]:
# Phase 2a — Dossier helpers: chamber regex, legal-area inference,
# doctrinal-density regex. These are general Swiss-legal-system primitives,
# NOT query-specific knowledge.

import re as _re_d

# ─── Statute-anchor canonicalizer ─────────────────────────────────────────────
# MIRROR of cell 12 in the v7.5 notebook. The corpus stores doc_statute_anchors
# in canonical form "41 OR" (not "Art. 41 OR"); we must canonicalize the
# LLM-produced ALL_TARGETS[qid]["statute_targets"] the same way before intersecting.

CODE_ALIAS = {
    "CPP": "StPO", "CP": "StGB", "CC": "ZGB", "CO": "OR",
    "LTF": "BGG", "LACI": "AVIG", "LAA": "UVG",
    "LP": "SchKG", "LDIP": "IPRG", "Cst": "BV", "Cst.": "BV",
    "STPO": "StPO", "OBG": "OR",
}
_ART_RE  = _re_d.compile(r"art\.?\s*(\d+[a-z]?)", _re_d.I)
_CODE_RE = _re_d.compile(r"\b([A-Z][A-Za-z]{1,8}\.?)\b")

def statute_anchor_canonical(raw):
    if not raw: return None
    s = str(raw).strip()
    m = _ART_RE.search(s)
    if not m: return None
    cands = [c.strip(".") for c in _CODE_RE.findall(s)
             if c.strip(".") not in ("Art", "Abs", "Ziff", "lit", "let", "al", "Bst")]
    if not cands: return None
    code = CODE_ALIAS.get(cands[-1], cands[-1])
    return f"{m.group(1)} {code}"

def canonicalize_target_statutes(targets):
    """Return canonical set matching doc_statute_anchors format."""
    out = set()
    for raw in (targets.get("statute_targets") or []):
        c = statute_anchor_canonical(raw)
        if c: out.add(c)
    return out

# ─── Chamber → coarse legal area (Swiss Federal Court structure) ───
CHAMBER_TO_AREA = {
    "I":   "public_law",
    "II":  "public_law",
    "III": "social_security",
    "IV":  "social_security",
    "V":   "civil",
    "VI":  "criminal",
    "1B":  "criminal_procedure",
    "1C":  "public_administrative",
    "2C":  "tax_administrative",
    "2D":  "tax_administrative",
    "4A":  "civil",
    "4F":  "civil",
    "5A":  "civil_family_succession",
    "5D":  "civil",
    "5F":  "civil",
    "6B":  "criminal",
    "6F":  "criminal",
    "7B":  "criminal",
    "8C":  "social_unemployment",
    "8D":  "social_unemployment",
    "9C":  "social_unemployment",
    "9F":  "social_unemployment",
}

_RE_CHAMBER_BGE = _re_d.compile(r"^BGE\s+\d+\s+([IVX]+)\s+\d+")
_RE_CHAMBER_BGR = _re_d.compile(r"^(\d[A-Z])_\d+")

def chamber_for_citation(citation):
    if not citation:
        return None
    m = _RE_CHAMBER_BGE.match(citation)
    if m: return m.group(1)
    m = _RE_CHAMBER_BGR.match(citation)
    if m: return m.group(1)
    return None

def chamber_class(chamber):
    return CHAMBER_TO_AREA.get(chamber) if chamber else None

# ─── Legal-area inference from a query's expansion (NOT hardcoded per query) ───
# Maps area → keyword set; we tally hits in the query expansion blob.
AREA_KEYWORDS = {
    "criminal_procedure":      ["pretrial", "detention", "kollusion", "kollusionsgefahr",
                                "stpo", "cpp", "untersuchung", "haft", "strafverfahren",
                                "procédure pénale", "détention provisoire"],
    "criminal":                ["strafgesetz", "criminal law", "stgb", "diritto penale",
                                "strafrecht", "code pénal"],
    "civil":                   ["civil law", "obligationenrecht", "art. or", " or ",
                                "contract", "obligation", "schaden", "haftung",
                                "responsabilité", "kaufvertrag", "code des obligations"],
    "civil_family_succession": ["zgb", "cc ", "civil code", "succession", "erbschaft",
                                "scheidung", "familienrecht", "marriage", "code civil"],
    "social_security":         ["ahv", "iv ", "atsg", "lpga", "lai", "invalidenversicherung",
                                "invalidity", "rente", "pension", "ivg", "social insurance"],
    "social_unemployment":     ["unemployment", "alv", "lacl", "arbeitslosen",
                                "assurance chômage"],
    "tax_administrative":      ["tax", "steuer", "dbg", "lifd", "impôt", "imposta"],
    "public_administrative":   ["verwaltungsverfahren", "administrative procedure", "rvog"],
    "public_law":              ["constitutional", "verfassung", "bv ", "cst ", "grundrecht",
                                "fundamental right", "freiheit"],
}

def legal_area_for_query(qid, targets, query_text):
    blob_parts = [(query_text or "").lower()]
    for k in ("legal_area_keywords", "concept_targets_en", "statute_targets"):
        v = (targets or {}).get(k) or []
        if v: blob_parts.extend(str(x).lower() for x in v)
    blob = " ".join(blob_parts)
    scores = {}
    for area, kws in AREA_KEYWORDS.items():
        scores[area] = sum(1 for kw in kws if kw in blob)
    best = max(scores, key=scores.get) if scores else "unknown"
    return best if scores.get(best, 0) > 0 else "unknown"

# ─── Doctrinal-density regex (DE / FR / IT) ───
# Empirically validated: gold court paras score 0.5-0.9, ambient 0.1-0.3, facts/dispositif < 0.1.

_RULE_OPENER_DE = _re_d.compile(r"(?:^|\s)(?:Nach|Gemäss|Im\s+Sinne\s+von)\s+Art\.\s*\d+")
_RULE_OPENER_FR = _re_d.compile(r"(?:^|\s)(?:Selon|Conformément\s+à|Aux\s+termes\s+de|En\s+vertu\s+de)\s+l?'?\s*art\.\s*\d+", _re_d.IGNORECASE)
_RULE_OPENER_IT = _re_d.compile(r"(?:^|\s)(?:Conformemente\s+all'|Ai\s+sensi\s+dell')\s*art\.\s*\d+", _re_d.IGNORECASE)
_DOCTRINE_DE    = _re_d.compile(r"\b(?:ständige\s+Rechtsprechung|nach\s+der\s+Rechtsprechung|Lehre\s+und\s+Rechtsprechung|Praxis\s+des\s+Bundesgerichts)\b", _re_d.IGNORECASE)
_DOCTRINE_FR    = _re_d.compile(r"\b(?:la\s+jurisprudence|selon\s+la\s+doctrine|il\s+est\s+constant|jurisprudence\s+constante)\b", _re_d.IGNORECASE)
_DOCTRINE_IT    = _re_d.compile(r"\b(?:la\s+giurisprudenza|secondo\s+la\s+dottrina)\b", _re_d.IGNORECASE)
_INTERNAL_CITE  = _re_d.compile(r"(?:ATF|BGE)\s+\d+\s+[IVX]+\s+\d+\s+(?:consid|E)\.\s*\d")
_RULE_VERB_DE   = _re_d.compile(r"Art\.\s*\d+[\w\s.,]{0,40}\b(?:bestimmt|sieht\s+vor|regelt|verlangt)\b", _re_d.IGNORECASE)
_RULE_VERB_FR   = _re_d.compile(r"art\.\s*\d+[\w\s.,]{0,40}\b(?:dispose|prévoit|précise|exige)\b", _re_d.IGNORECASE)
_HARD_NEG_DE    = _re_d.compile(r"\b(?:Sachverhalt|Verfahrensgeschichte|wird\s+abgewiesen|Gerichtskosten|der\s+Präsident|der\s+Gerichtsschreiber)\b", _re_d.IGNORECASE)
_HARD_NEG_FR    = _re_d.compile(r"\b(?:en\s+fait|le\s+recourant\s+fait\s+valoir|est\s+rejeté|frais\s+judiciaires|le\s+greffier)\b", _re_d.IGNORECASE)
_HARD_NEG_IT    = _re_d.compile(r"\b(?:in\s+fatto|spese\s+giudiziarie|è\s+respinto|il\s+cancelliere)\b", _re_d.IGNORECASE)
_STATUTE_REF    = _re_d.compile(r"\bart(?:icle)?\.?\s*\d+", _re_d.IGNORECASE)

def doctrinal_density(text, lang="de"):
    if not text or len(text) < 40:
        return 0.0
    lang = (lang or "de").lower()[:2]
    score = 0.30  # baseline
    if lang == "de":
        if _RULE_OPENER_DE.search(text): score += 0.30
        if _DOCTRINE_DE.search(text):    score += 0.20
        if _RULE_VERB_DE.search(text):   score += 0.15
        if _HARD_NEG_DE.search(text):    score -= 0.40
    elif lang == "fr":
        if _RULE_OPENER_FR.search(text): score += 0.30
        if _DOCTRINE_FR.search(text):    score += 0.20
        if _RULE_VERB_FR.search(text):   score += 0.15
        if _HARD_NEG_FR.search(text):    score -= 0.40
    elif lang == "it":
        if _RULE_OPENER_IT.search(text): score += 0.30
        if _DOCTRINE_IT.search(text):    score += 0.20
        if _HARD_NEG_IT.search(text):    score -= 0.40
    n_internal = len(_INTERNAL_CITE.findall(text))
    if n_internal >= 3:   score += 0.20
    elif n_internal >= 1: score += 0.05
    n_refs = len(_STATUTE_REF.findall(text))
    refs_per_1000 = n_refs / max(1, len(text) / 1000)
    score += min(0.15, refs_per_1000 * 0.05)
    return max(0.0, min(1.0, score))

def is_dispositif_or_facts(text, lang="de"):
    if not text: return False
    lang = (lang or "de").lower()[:2]
    if lang == "de" and _HARD_NEG_DE.search(text): return True
    if lang == "fr" and _HARD_NEG_FR.search(text): return True
    if lang == "it" and _HARD_NEG_IT.search(text): return True
    return False

def is_federal_court(court_base):
    if not court_base: return False
    cb = str(court_base).strip().upper()
    if cb.startswith("BGE") or cb.startswith("BGER"): return True
    if _RE_CHAMBER_BGR.match(cb): return True
    return False

# Substantive role set (boost) vs noise role set (penalize)
# IMPORTANT: NOISE_ROLES is narrow on purpose. v7.5 missed-gold diagnostic
# showed gold paragraphs with paragraph_role='facts' (BGE 148 V 21, BGE 140 V 193)
# — so we cannot blanket-drop "facts" / "procedural_history" / "neutral_default".
# Only the truly-never-gold classes (notification / costs / dispositif).
SUBSTANTIVE_ROLES = {"legal_standard", "reasoning", "application", "holding"}
NOISE_ROLES       = {"notification", "costs", "dispositif"}

# Sanity prints
print("[helpers] chamber_for_citation('BGE 142 III 296') =",
      chamber_for_citation("BGE 142 III 296"))
print("[helpers] chamber_for_citation('1B_490/2017')      =",
      chamber_for_citation("1B_490/2017"))
print("[helpers] chamber_class('1B') =", chamber_class("1B"))
print("[helpers] doctrinal_density sample DE:",
      round(doctrinal_density("Nach Art. 41 OR bestimmt das Bundesgericht in ständiger Rechtsprechung. Vgl. BGE 142 III 296 E. 4.1 und BGE 140 II 88 consid. 3.2.", "de"), 3))
print("[helpers] doctrinal_density sample DE facts:",
      round(doctrinal_density("Sachverhalt: Die Beschwerdeführerin macht geltend, die Vorinstanz habe Art. 95 BGG verletzt.", "de"), 3))


[helpers] chamber_for_citation('BGE 142 III 296') = III
[helpers] chamber_for_citation('1B_490/2017')      = 1B
[helpers] chamber_class('1B') = criminal_procedure
[helpers] doctrinal_density sample DE: 0.85
[helpers] doctrinal_density sample DE facts: 0.0


## 2b. Compute the dossier per (query, candidate)


In [5]:
# Phase 2b — Dossier compute: per-(qid, did) cross features
# Builds PER_QUERY[qid]["dossier"][did] = {feature_dict}.
# Uses helpers from _dossier_helpers cell + global state from warm-boot.

DOSSIER_TOP_N = 50000  # use the FULL snapshot per-query pool

print("[dossier] computing per-doc cache for", len(doc_meta), "unique docs...")
_t0 = time.time()
_doc_chamber       = {}
_doc_chamber_cls   = {}
_doc_doctrinal     = {}
_doc_hard_neg      = {}
_doc_is_federal    = {}
_doc_lead_anchors  = {}

for did, meta in doc_meta.items():
    cit  = meta.get("citation") or did
    ch   = chamber_for_citation(cit)
    _doc_chamber[did]     = ch
    _doc_chamber_cls[did] = chamber_class(ch)
    _doc_is_federal[did]  = is_federal_court(meta.get("court_base", "")) or (ch is not None)
    text = (search_text.get(did, "") or "")
    lang = (meta.get("language") or "de").lower()[:2]
    _doc_doctrinal[did]   = doctrinal_density(text, lang)
    _doc_hard_neg[did]    = is_dispositif_or_facts(text, lang)
    # Lead-anchors: which of this doc's statute_anchors appear in first 200 chars
    lead = text[:200].lower()
    anchors = doc_statute_anchors.get(did, set())
    lead_set = set()
    for a in anchors:
        a_low = a.lower()
        if a_low in lead:
            lead_set.add(a); continue
        # Fallback: look for the article-number stem (handles minor formatting drift)
        m = re.match(r"art\.?\s*\d+\w*", a_low)
        if m and m.group(0) in lead:
            lead_set.add(a)
    _doc_lead_anchors[did] = lead_set
print(f"  per-doc cache built in {time.time()-_t0:.1f}s")

# Per-query cross features
print("[dossier] computing per-(qid, did) cross features...")
for q in ALL_QUERIES:
    qid     = q["query_id"]
    qtext   = q["query_text"]
    targets = ALL_TARGETS.get(qid, {}) or {}
    aspects = ALL_HYDE_ASPECTS.get(qid, []) or []

    pool = PER_QUERY[qid].get("final_topk", [])[:DOSSIER_TOP_N]

    # CANONICALIZE statute targets to match doc_statute_anchors format ("41 OR" not "Art. 41 OR")
    target_statutes = canonicalize_target_statutes(targets)
    target_concepts = set(targets.get("concept_targets_en", []) or [])
    target_terms_de = set(targets.get("term_targets_de", []) or [])
    target_terms_fr = set(targets.get("term_targets_fr", []) or [])
    target_terms_it = set(targets.get("term_targets_it", []) or [])

    # Aspect keyword bags (one set per aspect)
    aspect_kw = []
    for a in aspects:
        atxt = a.get("paragraph") or a.get("answer") or str(a) if isinstance(a, dict) else str(a)
        toks = set(t.lower() for t in re.findall(r"[A-Za-zÄÖÜäöüß]{4,}", atxt))
        aspect_kw.append(toks)

    legal_area = legal_area_for_query(qid, targets, qtext)

    # Co-citation: for each LAW in pool, count COURT paras in pool whose anchors cite it
    law_cit_to_did = {}
    for did in pool:
        m = doc_meta.get(did, {})
        if m.get("family") == "law":
            cit = (m.get("citation") or "").strip()
            if cit: law_cit_to_did[cit] = did
    co_cite = Counter()
    for did in pool:
        m = doc_meta.get(did, {})
        if m.get("family") != "court": continue
        for sa in doc_statute_anchors.get(did, set()):
            tgt = law_cit_to_did.get(sa)
            if tgt: co_cite[tgt] += 1

    # Case peers: court paras sharing court_base
    case_peer = defaultdict(list)
    for did in pool:
        m = doc_meta.get(did, {})
        if m.get("family") == "court":
            cb = (m.get("court_base") or "").strip()
            if cb: case_peer[cb].append(did)

    dossier = {}
    for did in pool:
        m       = doc_meta.get(did, {})
        family  = m.get("family", "?")
        lang    = (m.get("language") or "?").lower()[:2]
        anchors = doc_statute_anchors.get(did, set())
        concs   = _doc_to_concepts.get(did, set())

        stat_full = anchors & target_statutes
        stat_lead = _doc_lead_anchors.get(did, set()) & target_statutes
        conc_ovl  = concs & target_concepts

        if   lang == "de": term_ovl = _doc_to_terms.get(did, set()) & target_terms_de
        elif lang == "fr": term_ovl = _doc_to_terms.get(did, set()) & target_terms_fr
        elif lang == "it": term_ovl = _doc_to_terms.get(did, set()) & target_terms_it
        else:              term_ovl = set()

        # Aspect-best-match
        if aspect_kw:
            doc_text_low = (search_text.get(did, "") or "")[:1500].lower()
            anchor_blob  = " ".join(anchors).lower() + " " + " ".join(concs).lower()
            blob_toks    = set(t for t in re.findall(r"[A-Za-zÄÖÜäöüß]{4,}", doc_text_low + " " + anchor_blob))
            ascores      = [len(akw & blob_toks) for akw in aspect_kw]
            best_aspect  = max(range(len(ascores)), key=lambda i: ascores[i])
            aspect_score = ascores[best_aspect]
            n_addressed  = sum(1 for s in ascores if s >= 2)
        else:
            best_aspect, aspect_score, n_addressed = 0, 0, 0

        ch_cls   = _doc_chamber_cls.get(did)
        ch_match = (ch_cls is not None and ch_cls == legal_area)

        cb    = (m.get("court_base") or "").strip()
        peers = case_peer.get(cb, [])
        cp_n  = max(0, len(peers) - 1)
        cp_rule = any(
            doc_meta.get(p, {}).get("paragraph_role") in SUBSTANTIVE_ROLES
            for p in peers if p != did
        )

        dossier[did] = {
            "family":            family,
            "language":          lang,
            "chamber":           _doc_chamber.get(did),
            "chamber_class":     ch_cls,
            "is_federal":        _doc_is_federal.get(did, False),
            "paragraph_role":    m.get("paragraph_role", ""),
            "stat_overlap_full": sorted(stat_full),
            "stat_overlap_lead": sorted(stat_lead),
            "conc_overlap":      sorted(conc_ovl),
            "term_overlap":      sorted(term_ovl),
            "best_aspect":       int(best_aspect),
            "aspect_score":      int(aspect_score),
            "n_aspects_addressed": int(n_addressed),
            "chamber_match":     bool(ch_match),
            "legal_area":        legal_area,
            "co_cite_count":     int(co_cite.get(did, 0)),
            "case_peer_count":   int(cp_n),
            "case_peer_rule":    bool(cp_rule),
            "doctrinal":         round(float(_doc_doctrinal.get(did, 0.0)), 3),
            "is_dispositif":     bool(_doc_hard_neg.get(did, False)),
        }

    PER_QUERY[qid]["dossier"]    = dossier
    PER_QUERY[qid]["legal_area"] = legal_area

print(f"[dossier] complete for {len(PER_QUERY)} queries in {time.time()-_t0:.1f}s\n")
print(f"{'qid':<10}{'pool':>7}{'avg_stat':>10}{'avg_lead':>10}{'ch_match':>10}{'disp':>8}{'fed_court':>11}  legal_area")
for qid in sorted(PER_QUERY):
    d = PER_QUERY[qid]["dossier"]
    n = len(d)
    if not n: continue
    avg_stat = sum(len(x["stat_overlap_full"]) for x in d.values()) / n
    avg_lead = sum(len(x["stat_overlap_lead"]) for x in d.values()) / n
    n_chm    = sum(1 for x in d.values() if x["chamber_match"])
    n_disp   = sum(1 for x in d.values() if x["is_dispositif"])
    n_fed_ct = sum(1 for x in d.values() if x["family"] == "court" and x["is_federal"])
    print(f"{qid:<10}{n:>7}{avg_stat:>10.2f}{avg_lead:>10.2f}{n_chm:>10}{n_disp:>8}{n_fed_ct:>11}  {PER_QUERY[qid].get('legal_area','?')}")


[dossier] computing per-doc cache for 255713 unique docs...
  per-doc cache built in 5.4s
[dossier] computing per-(qid, did) cross features...
[dossier] complete for 10 queries in 25.8s

qid          pool  avg_stat  avg_lead  ch_match    disp  fed_court  legal_area
val_001     44621      0.20      0.01     10640       0      38350  criminal_procedure
val_002     41254      0.08      0.00       326       0      27087  social_security
val_003     44033      0.20      0.01     10798       0      37623  criminal_procedure
val_004     43086      0.04      0.00     13341       0      34932  civil_family_succession
val_005     42140      0.00      0.00      3298       0      33939  criminal
val_006     46925      0.18      0.02      8772       0      37660  civil
val_007     40953      0.02      0.00     11774       0      34322  civil_family_succession
val_008     45818      0.13      0.01     14872       0      37891  criminal
val_009     40751      0.03      0.00      6244       0      335

## Stage A — Cheap rule filter (no LLM)

Drops candidates that are *structurally* incompatible with being gold based on cheap features:
- Cantonal courts (102/102 val gold are Federal Supreme Court)
- Facts / procedural_history / notification / costs / dispositif paragraph roles
- Hard-negative regex hits ("Sachverhalt", "en fait", "wird abgewiesen", "le greffier", ...)
- Zero signal: no statute / concept / term overlap AND not co-cited

Expected reduction: ~60-85% of pool, ~1% gold loss.


In [6]:
# Stage A — cheap rule-based prune (no LLM)
# Drops candidates that are structurally unable to be gold based on cheap signals.
# Typical reduction: 60-85% of pool, < 1% gold loss.

print("[Stage A] applying cheap filters...")
for q in ALL_QUERIES:
    qid     = q["query_id"]
    pool    = PER_QUERY[qid].get("final_topk", [])[:DOSSIER_TOP_N]
    dossier = PER_QUERY[qid].get("dossier", {})

    kept   = []
    drops  = Counter()

    for did in pool:
        d = dossier.get(did)
        if d is None:
            drops["no_dossier"] += 1; continue

        # Rule 1 — cantonal courts are never gold (102/102 val gold are federal)
        if d["family"] == "court" and not d["is_federal"]:
            drops["cantonal_court"] += 1; continue

        # Rule 2 — paragraph_role noise classes
        if d["paragraph_role"] in NOISE_ROLES:
            drops["noise_role"] += 1; continue

        # Rule 3 — hard-negative regex hit (dispositif / facts / notification language)
        if d["is_dispositif"]:
            drops["dispositif_regex"] += 1; continue

        # NOTE: removed the "no_signal" rule. After canonicalization fix, most
        # candidates have some overlap, but some gold (especially law-side bridge
        # articles) have zero direct signal and rely on channel-based discovery.
        # Stage B's composite score + top-N cap handles the ranking.

        kept.append(did)

    PER_QUERY[qid]["stage_a_kept"] = kept

    _g_set       = ALL_GOLD_DOC_SET.get(qid, set())
    gold_before  = len(set(pool) & _g_set)
    gold_after   = len(set(kept) & _g_set)
    pct_kept     = len(kept) / max(1, len(pool)) * 100
    pct_gold_kept= gold_after / max(1, gold_before) * 100 if gold_before else 100.0
    print(f"  [{qid}] {len(pool):>5} → {len(kept):>5} ({pct_kept:5.1f}%)  "
          f"gold {gold_before}→{gold_after} ({pct_gold_kept:5.1f}%)  drops: {dict(drops)}")


[Stage A] applying cheap filters...
  [val_001] 44621 → 40809 ( 91.5%)  gold 39→39 (100.0%)  drops: {'cantonal_court': 2421, 'noise_role': 1391}
  [val_002] 41254 → 29418 ( 71.3%)  gold 30→30 (100.0%)  drops: {'cantonal_court': 10074, 'noise_role': 1762}
  [val_003] 44033 → 40198 ( 91.3%)  gold 36→36 (100.0%)  drops: {'cantonal_court': 2430, 'noise_role': 1405}
  [val_004] 43086 → 37644 ( 87.4%)  gold 10→10 (100.0%)  drops: {'cantonal_court': 4178, 'noise_role': 1264}
  [val_005] 42140 → 37406 ( 88.8%)  gold 11→11 (100.0%)  drops: {'cantonal_court': 3595, 'noise_role': 1139}
  [val_006] 46925 → 40168 ( 85.6%)  gold 17→17 (100.0%)  drops: {'cantonal_court': 5402, 'noise_role': 1355}
  [val_007] 40953 → 36807 ( 89.9%)  gold 17→17 (100.0%)  drops: {'cantonal_court': 2761, 'noise_role': 1385}
  [val_008] 45818 → 41157 ( 89.8%)  gold 25→25 (100.0%)  drops: {'cantonal_court': 3515, 'noise_role': 1146}
  [val_009] 40751 → 35828 ( 87.9%)  gold 13→13 (100.0%)  drops: {'cantonal_court': 3164, 'n

## Stage B — Composite dossier score + top-500 per query

Linear combination of 7+ dossier features. Lead-statute intersection has the highest weight (research found it's the single sharpest discriminator). Chamber-match adds +2. Substantive paragraph_role adds +1.5. Hard-negative penalties are already applied in Stage A.

Output: top-500 per query handed to the LLM.


In [7]:
# Stage B — Fusion-rank ranking (TRUST the v7.5 RRF fusion order)
#
# WHY this design: Earlier dossier-based composite scoring achieved R@300 = 0.013
# because most val gold has ZERO statute-target intersection (LLM expansion covers
# 10-15 statutes per query, but gold spans 40+). The 14-channel RRF fusion in v7.5
# already aggregates statute + graph + vector + BM25 + concept signals and achieves
# R@5000 = 0.728 macro — far better than any single dossier signal we can compute
# without channel-of-arrival info (which the snapshot doesn't preserve).
#
# So: walk fusion order, keep the first N that survived Stage A. The composite
# score is still computed (Stage E uses it for tiebreaks; Stage C/D show dossier
# in the LLM prompts) but it's NOT used to reorder candidates here.

STAGE_B_TOP_N = 2000  # was 300 — bigger pool for higher recall ceiling
                       # fusion R@2000 ≈ 0.62 macro; R@5000 ≈ 0.73

# Composite score retained — used by Stage E for tiebreaks (NOT for ranking)
def _composite_score(d):
    s  = 0.0
    s += 8.0 * len(d["stat_overlap_lead"])
    s += 3.0 * (len(d["stat_overlap_full"]) - len(d["stat_overlap_lead"]))
    s += 2.0 * len(d["conc_overlap"])
    s += 1.0 * len(d["term_overlap"])
    s += 1.5 if d["chamber_match"] else 0.0
    s += 1.0 if d["paragraph_role"] in SUBSTANTIVE_ROLES else 0.0
    s += 0.5 * d["doctrinal"]
    s += 0.1 * min(d["aspect_score"], 10)
    s += 0.3 * min(d["co_cite_count"], 20)
    s += 0.5 if d.get("case_peer_rule") else 0.0
    return s

print(f"[Stage B] fusion-rank ranking + top-{STAGE_B_TOP_N} per query")
print(f"  Trusting the v7.5 RRF order from PER_QUERY[qid]['final_topk'].")
print(f"  Dossier score still computed for downstream display/tiebreaks.\n")

for q in ALL_QUERIES:
    qid       = q["query_id"]
    fusion    = PER_QUERY[qid].get("final_topk", [])     # already RRF-ranked
    kept_set  = set(PER_QUERY[qid].get("stage_a_kept", []))
    dossier   = PER_QUERY[qid]["dossier"]
    _g_set    = ALL_GOLD_DOC_SET.get(qid, set())

    # Walk fusion order, keep first N that survived Stage A
    top = []
    seen = set()
    for did in fusion:
        if did in kept_set and did not in seen:
            top.append((did, _composite_score(dossier[did])))
            seen.add(did)
            if len(top) >= STAGE_B_TOP_N:
                break

    PER_QUERY[qid]["stage_b_ranked"]    = top
    PER_QUERY[qid]["stage_b_score_map"] = dict(top)

    # Diagnostic — where does gold actually live in fusion rank?
    gold_ranks = [i for i, did in enumerate(fusion) if did in _g_set]
    if gold_ranks:
        gr_min = min(gold_ranks)
        gr_med = sorted(gold_ranks)[len(gold_ranks)//2]
        gr_max = max(gold_ranks)
        gr_in_topN = sum(1 for r in gold_ranks if r < STAGE_B_TOP_N)
    else:
        gr_min, gr_med, gr_max, gr_in_topN = 0, 0, 0, 0

    gold_in_top = len({d for d, _ in top} & _g_set)
    R_at_b      = gold_in_top / max(1, len(_g_set))

    print(f"  [{qid}]  fusion={len(fusion):>5}  kept={len(kept_set):>5}  top-{len(top):<4}"
          f"  gold {gold_in_top:>3}/{len(_g_set):<3}  R={R_at_b:.3f}")
    print(f"         gold fusion ranks: min={gr_min:>5}  median={gr_med:>5}  max={gr_max:>5}  "
          f"in top-{STAGE_B_TOP_N}={gr_in_topN}/{len(_g_set)}")

# Macro mean R
_macro_r = sum(
    len({d for d,_ in PER_QUERY[q['query_id']]['stage_b_ranked']} & ALL_GOLD_DOC_SET.get(q['query_id'], set())) /
    max(1, len(ALL_GOLD_DOC_SET.get(q['query_id'], set())))
    for q in ALL_QUERIES
) / len(ALL_QUERIES)
print(f"\n[Stage B] macro mean R@{STAGE_B_TOP_N}: {_macro_r:.3f}")
print(f"          (fusion-baseline R@5000 = 0.728; R@1000 = 0.530; R@500 = 0.415)")


[Stage B] fusion-rank ranking + top-2000 per query
  Trusting the v7.5 RRF order from PER_QUERY[qid]['final_topk'].
  Dossier score still computed for downstream display/tiebreaks.

  [val_001]  fusion=44621  kept=40809  top-2000  gold  16/42   R=0.381
         gold fusion ranks: min=   58  median= 2438  max=41138  in top-2000=16/42
  [val_002]  fusion=41254  kept=29418  top-2000  gold  22/38   R=0.579
         gold fusion ranks: min=    2  median=  355  max=37532  in top-2000=21/38
  [val_003]  fusion=44033  kept=40198  top-2000  gold  14/47   R=0.298
         gold fusion ranks: min=   37  median= 3682  max=41783  in top-2000=12/47
  [val_004]  fusion=43086  kept=37644  top-2000  gold   9/10   R=0.900
         gold fusion ranks: min=   58  median=  427  max= 7617  in top-2000=9/10
  [val_005]  fusion=42140  kept=37406  top-2000  gold   9/11   R=0.818
         gold fusion ranks: min=   49  median=  182  max=11045  in top-2000=9/11
  [val_006]  fusion=46925  kept=40168  top-2000  gold  

# Phase 3 — LLM Cascade


## 3a. Load Qwen3-32B (used by Stages C, D, F)


In [8]:
# Phase 3a — Load Qwen3-32B-AWQ via vLLM (SM 12.x-aware; no parent-process CUDA init)
#
# Lessons baked in from previous Blackwell SM 120 runs:
#   1. NEVER call torch.cuda.* in this cell before vLLM loads. Doing so initializes
#      CUDA in the parent process and breaks vLLM's V1 worker handshake — produces
#      the silent "Failed core proc(s): {}" crash.
#   2. FlashInfer 0.6.x has NO kernels for SM 12.x (consumer/workstation Blackwell).
#      Python import succeeds, kernel launch crashes worker. Detect via nvidia-smi
#      and force FLASHINFER_USABLE = False on SM 12.x.
#   3. If load fails: RESTART THE RUNTIME. Don't retry in-process — leftover CUDA
#      state poisons subsequent attempts.

import os as _os_l
import gc as _gc_l
import subprocess as _sub_l
import inspect as _inspect_l

# Set vLLM worker start method BEFORE any other vLLM-related action
_os_l.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

# Free upstream symbols — but DO NOT touch torch.cuda.* here
for _name in ("EMB_MODEL", "E_GPU", "rrk_mod", "rrk_tok"):
    if _name in globals():
        try: del globals()[_name]
        except KeyError: pass
_gc_l.collect()

# GPU detection via nvidia-smi (does NOT init CUDA in this process)
GPU_SM = 0
GPU_NAME = "?"
try:
    _out = _sub_l.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10, check=True,
    )
    line = _out.stdout.strip().splitlines()[0]
    GPU_NAME, _cc = [x.strip() for x in line.split(",")[:2]]
    _maj, _min = _cc.split(".")
    GPU_SM = int(_maj) * 10 + int(_min)
    print(f"[llm] GPU: {GPU_NAME}  (compute capability {_cc} = SM {GPU_SM})")
except Exception as e:
    print(f"[llm] nvidia-smi probe failed: {e!r} — proceeding without SM info")

# FlashInfer detection — install check only, no kernel call
HAS_FLASHINFER = False
try:
    import flashinfer as _fi
    HAS_FLASHINFER = True
    print(f"[llm] FlashInfer {getattr(_fi, '__version__', '?')} importable")
except ImportError:
    print("[llm] FlashInfer not importable")

# Decide if FlashInfer is USABLE on this GPU.
# SM 12.x (RTX 5090 / RTX PRO 6000 consumer/workstation Blackwell) has no pre-built
# kernels in FlashInfer 0.6.x. Forcing FLASHINFER on SM 12.x crashes the worker.
FLASHINFER_USABLE = HAS_FLASHINFER and (GPU_SM // 10 != 12)
if HAS_FLASHINFER and not FLASHINFER_USABLE:
    print(f"[llm] SM {GPU_SM} → FlashInfer DISABLED (no kernels for SM 12.x in 0.6.x).")
    print("[llm]   vLLM will auto-select FlashAttention 4 — equally fast on Blackwell.")
elif FLASHINFER_USABLE:
    print(f"[llm] SM {GPU_SM} → requesting FLASHINFER backend")
else:
    print("[llm] vLLM will auto-select attention backend")

# Config
LLM_MODEL              = "Qwen/Qwen3-32B-AWQ"
MAX_MODEL_LEN          = 4096
MAX_NUM_SEQS           = 256
GPU_MEM_UTIL           = 0.90
LLM_TEMPERATURE        = 0.1
LLM_TOP_P              = 0.9
LLM_REPETITION_PENALTY = 1.05

from vllm import LLM as _VLLM, SamplingParams as _VSP
from transformers import AutoTokenizer as _AT

print(f"\n[llm] loading {LLM_MODEL} via vLLM...")
_t0 = time.time()

llm_tok = _AT.from_pretrained(LLM_MODEL, trust_remote_code=True)

# Probe prefix size — diagnostic only
_PROBE_SYS = "You are a Swiss legal citation expert. Score 1-5."
_probe = llm_tok.apply_chat_template(
    [{"role": "system", "content": _PROBE_SYS}, {"role": "user", "content": "probe"}],
    tokenize=False, add_generation_prompt=True,
)
_probe_n = len(llm_tok(_probe, add_special_tokens=False)["input_ids"])
print(f"[llm] system+template prefix ≈ {_probe_n} tokens (prefix-cached)")

# Build kwargs — pass everything via **kwargs (the inspect.signature trick of the
# earlier court notebook silently dropped real fields; pass them directly).
sig = set(_inspect_l.signature(_VLLM).parameters.keys())
llm_kwargs = dict(
    model=LLM_MODEL,
    trust_remote_code=True,
    tensor_parallel_size=1,
    gpu_memory_utilization=GPU_MEM_UTIL,
    max_model_len=MAX_MODEL_LEN,
    max_num_seqs=MAX_NUM_SEQS,
    enforce_eager=False,
    disable_custom_all_reduce=True,
    disable_log_stats=True,
    quantization="awq_marlin",
    enable_prefix_caching=True,
)

_backend_method = "auto-select"
if FLASHINFER_USABLE:
    if "attention_backend" in sig:
        llm_kwargs["attention_backend"] = "FLASHINFER"
        _backend_method = "attention_backend=FLASHINFER (kwarg)"
    elif "attention_config" in sig:
        try:
            from vllm.config import AttentionConfig
            llm_kwargs["attention_config"] = AttentionConfig(backend="FLASHINFER")
            _backend_method = "attention_config=AttentionConfig(FLASHINFER)"
        except Exception:
            _os_l.environ["VLLM_ATTENTION_BACKEND"] = "FLASHINFER"
            _backend_method = "env VLLM_ATTENTION_BACKEND=FLASHINFER (deprecated)"
    else:
        _os_l.environ["VLLM_ATTENTION_BACKEND"] = "FLASHINFER"
        _backend_method = "env VLLM_ATTENTION_BACKEND=FLASHINFER (deprecated)"
    _os_l.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "1")

print(f"[llm] init plan:")
print(f"  attention backend  : {_backend_method}")
print(f"  quantization       : awq_marlin")
print(f"  prefix caching     : on")
print(f"  CUDA graphs        : on (enforce_eager=False)")
print(f"  max_model_len      : {MAX_MODEL_LEN}")
print(f"  max_num_seqs       : {MAX_NUM_SEQS}")
print(f"  gpu_mem_util       : {GPU_MEM_UTIL}")

# SINGLE load attempt. If it fails: RESTART RUNTIME and try with diagnostics.
try:
    llm_mod = _VLLM(**llm_kwargs)
except Exception as e:
    print(f"\n[llm] LOAD FAILED: {type(e).__name__}: {e}")
    print()
    print("Do NOT retry in-process — CUDA state is now poisoned.")
    print("Restart the Colab/Kaggle runtime, then try the fixes below in order:")
    print(f"  1. Your GPU is SM {GPU_SM}. If FLASHINFER_USABLE was True above, force it False.")
    print(f"  2. If still failing, drop GPU_MEM_UTIL = 0.85 and MAX_NUM_SEQS = 192.")
    print(f"  3. As a last resort: enforce_eager=True (skips graph capture, ~10-20% slower).")
    print(f"  4. For deep debug, set env VLLM_LOGGING_LEVEL=DEBUG and CUDA_LAUNCH_BLOCKING=1")
    print(f"     BEFORE this cell — surfaces the worker stderr that's currently hidden.")
    raise

print(f"\n[llm] loaded in {time.time()-_t0:.1f}s")

# NOW it's safe to import torch (vLLM has already initialized CUDA)
import torch as _torch_l
if _torch_l.cuda.is_available():
    for i in range(_torch_l.cuda.device_count()):
        free, total = _torch_l.cuda.mem_get_info(i)
        print(f"  GPU {i}: free={free/1024**3:.2f} GiB / {total/1024**3:.2f} GiB")

# ─── Generation API ────────────────────────────────────────────────────────
def _render_prompt(system, user):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        return llm_tok.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        return llm_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def _generate_safe(prompts, params, min_split=1):
    """Recursive split-on-failure for OOM. Validation errors are NOT retried:
    splitting the batch doesn't make any single prompt shorter, so retrying just
    re-fails. Returns "" for any prompt that exceeds context (Stage C, D, F
    callers handle empty responses as low-confidence with default scores)."""
    try:
        outs = llm_mod.generate(prompts, sampling_params=params, use_tqdm=False)
        return [(o.outputs[0].text if o.outputs else "") for o in outs]
    except Exception as e:
        ename = type(e).__name__
        # Validation errors: don't split — return empty strings for the offending batch.
        # The caller decides how to handle empties (Stage C falls back to score=1).
        if "ValidationError" in ename or "max_input" in str(e) or "context length" in str(e):
            if len(prompts) == 1:
                print(f"  [llm] prompt too long for context ({ename}); returning empty.")
                return [""]
            # For multi-prompt batches with one bad prompt, bisect so good ones still run.
            mid = len(prompts) // 2
            return _generate_safe(prompts[:mid], params, min_split) + \
                   _generate_safe(prompts[mid:], params, min_split)
        # Genuine OOM-style errors: split, free memory, retry the halves.
        if len(prompts) <= min_split: raise
        mid = len(prompts) // 2
        print(f"  [llm] batch of {len(prompts)} failed ({ename}); split {mid}+{len(prompts)-mid}")
        _gc_l.collect()
        if _torch_l.cuda.is_available(): _torch_l.cuda.empty_cache()
        return _generate_safe(prompts[:mid], params, min_split) + \
               _generate_safe(prompts[mid:], params, min_split)

def llm_generate(system, user, max_new_tokens=700, temperature=None):
    return llm_generate_batch(
        [(system, user)], max_new_tokens=max_new_tokens, temperature=temperature
    )[0]

def llm_generate_batch(messages_pairs, max_new_tokens=700, temperature=None):
    prompts = [_render_prompt(s, u) for s, u in messages_pairs]
    params = _VSP(
        temperature=LLM_TEMPERATURE if temperature is None else temperature,
        top_p=LLM_TOP_P,
        max_tokens=max_new_tokens,
        repetition_penalty=LLM_REPETITION_PENALTY,
    )
    return _generate_safe(prompts, params)

# Warm-up
print("[llm] warm-up (16 short prompts)...")
_t = time.time()
_warm = llm_generate_batch(
    [("Reply with the user's requested text and nothing else.",
      f"Reply with exactly: probe-{i}") for i in range(16)],
    max_new_tokens=20,
)
print(f"[llm] warm-up done in {time.time()-_t:.2f}s; sample: {_warm[0]!r}")

[llm] GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition  (compute capability 12.0 = SM 120)
[llm] FlashInfer 0.6.8.post1 importable
[llm] SM 120 → FlashInfer DISABLED (no kernels for SM 12.x in 0.6.x).
[llm]   vLLM will auto-select FlashAttention 4 — equally fast on Blackwell.

[llm] loading Qwen/Qwen3-32B-AWQ via vLLM...
[llm] system+template prefix ≈ 28 tokens (prefix-cached)
[llm] init plan:
  attention backend  : auto-select
  quantization       : awq_marlin
  prefix caching     : on
  CUDA graphs        : on (enforce_eager=False)
  max_model_len      : 4096
  max_num_seqs       : 256
  gpu_mem_util       : 0.9
INFO 05-13 01:56:21 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 256, 'disable_log_stats': True, 'quantization': 'awq_marlin', 'disable_custom_all_reduce': True, 'model': 'Qwen/Qwen3-32B-AWQ'}
INFO 05-13 01:56:22 [model.py:555] Resolved architecture: Qwen3ForC

Parse safetensors files:   0%|          | 0/4 [00:00<?, ?it/s]

INFO 05-13 01:56:23 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-13 01:56:23 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])

[llm] loaded in 42.7s
  GPU 0: free=10.25 GiB / 94.97 GiB
[llm] warm-up (16 short prompts)...
[llm] warm-up done in 3.12s; sample: 'probe-0'


## Stage C — Listwise LLM scoring with dossier (Qwen3-32B)

Each batch of 15 candidates is shown with its full Phase-1 dossier (cited statutes ∩ query targets, concepts caught, addressed aspect, chamber match, doctrinal density, co-citation count) plus a 400-char text excerpt. The LLM scores 1-5 and tags `addresses_aspect`. Score ≥ 3 survives to Stage D.

The dossier *changes the LLM's task*: instead of "does this look relevant from the text?", it becomes "does the text confirm the dossier's pre-computed evidence?" — much harder to hallucinate around.


In [ ]:
# Stage C — Listwise LLM scoring with the full Phase-1 dossier
# Per batch of 15 candidates: dossier signals + 400-char text excerpt.
# LLM outputs [{id, score 1-5, addresses_aspect, why}, ...].
# Keep candidates scoring ≥ 3 for Stage D.

import json as _json_c

STAGE_C_BATCH = 12                  # was 15 — slightly smaller for prompt size
STAGE_C_KEEP_THRESHOLD = 3
STAGE_C_MAX_NEW_TOKENS = 700        # was 1500 — output is ~12 items × ~30 tokens = 360, so 700 is enough

S2_SYSTEM = (
    "You are a Swiss legal citation expert. For each candidate, judge how strongly "
    "it answers the English legal question. Score 1-5:\n"
    "  5 = THE controlling statute or leading case for this question\n"
    "  4 = clearly relevant — applies the rule or supports a key sub-question\n"
    "  3 = related but peripheral — context, not the answer\n"
    "  2 = tangentially related — same topic area, wrong issue\n"
    "  1 = unrelated, false positive\n\n"
    "DOMAIN FACTS:\n"
    "- Documents are German / French / Italian. Multilingual is expected.\n"
    "- Swiss code aliases (treat as same code): CC=ZGB, CO=OR, CP=StGB, CPP=StPO, "
    "LP=SchKG, LIFD=DBG, LPGA=ATSG, LAI=IVG, LTF=BGG, LAA=UVG, Cst=BV.\n"
    "- Federal Supreme Court chambers signal legal area: 1B=criminal procedure, "
    "4A=civil, 5A=family/succession, 6B=criminal substantive, 7B=new criminal "
    "chamber, 8C=social/unemployment, 9C=social, II-IV=public/social BGE chambers.\n"
    "- Cantonal courts are not gold. Procedural-history, facts, dispositif, costs, "
    "and notification paragraphs are not gold even if they mention the right statute.\n\n"
    "USE THE DOSSIER. Each candidate has a [signals] block with PRE-COMPUTED evidence: "
    "lead-rule statute hits, concept overlap, addresses_aspect tag, chamber match, "
    "co-citation in pool, doctrinal-density score. Trust signals + verify against text. "
    "A candidate with 3+ lead-statute hits, chamber match, doctrinal density > 0.6, and "
    "a rule-recital opening is almost certainly score 4-5. A candidate with only "
    "concept overlap and no statute hits is at most 2-3.\n\n"
    "OUTPUT: strict JSON array. One object per candidate. No preamble. Schema:\n"
    '[{"id": <int>, "score": <1-5>, "addresses_aspect": <int>, "why": "<≤20 words>"}, ...]'
)

def _fmt_list(lst, n=4):
    if not lst: return "(none)"
    if len(lst) <= n: return ", ".join(lst)
    return ", ".join(lst[:n]) + f" +{len(lst)-n}"

def _c_format(idx, did, d, text_400, t_stat_n, t_conc_n, aspect_labels):
    cit  = doc_meta.get(did, {}).get("citation") or did
    role = d["paragraph_role"] or "n/a"
    ch_part = (f"chamber {d['chamber']} ({d['chamber_class']})"
               if d["chamber"] else "(no chamber)")
    ch_tag  = " ← matches query area" if d["chamber_match"] else ""
    if aspect_labels and 0 <= d["best_aspect"] < len(aspect_labels):
        asp_str = aspect_labels[d["best_aspect"]][:60]
    else:
        asp_str = "?"
    full_only = [s for s in d["stat_overlap_full"] if s not in d["stat_overlap_lead"]]
    return (
        f"\n[{idx}] {cit}  ({d['language']}, {d['family']}, {ch_part}{ch_tag}, role={role})\n"
        f"  cites in lead: {{{_fmt_list(d['stat_overlap_lead'])}}}  "
        f"+ rest: {{{_fmt_list(full_only)}}}   "
        f"← {len(d['stat_overlap_full'])}/{t_stat_n} query statutes\n"
        f"  concepts caught: {{{_fmt_list(d['conc_overlap'])}}}  "
        f"← {len(d['conc_overlap'])}/{t_conc_n}\n"
        f"  best aspect: #{d['best_aspect']+1} \"{asp_str}…\" (strength {d['aspect_score']})\n"
        f"  pool: cited_by={d['co_cite_count']} court paras · "
        f"case_peers={d['case_peer_count']} (rule_peer={d['case_peer_rule']})\n"
        f"  doctrinal_density={d['doctrinal']:.2f}\n"
        f"  text: {text_400}\n"
    )

def _c_parse(resp, n_expected):
    m = re.search(r"\[.*\]", resp, re.S)
    if not m: return {}
    try: arr = _json_c.loads(m.group(0))
    except Exception: return {}
    if not isinstance(arr, list): return {}
    out = {}
    for item in arr:
        if not isinstance(item, dict): continue
        try:
            idx   = int(item.get("id", -1))
            if not (1 <= idx <= n_expected): continue
            score = max(1, min(5, int(item.get("score", 0))))
            asp   = max(0, int(item.get("addresses_aspect", 1)) - 1)
            why   = str(item.get("why", ""))[:200]
            out[idx] = {"score": score, "addresses_aspect": asp, "why": why}
        except (TypeError, ValueError):
            continue
    return out

print(f"[Stage C] listwise scoring on Stage-B top-{STAGE_B_TOP_N} per query (batch={STAGE_C_BATCH})...")
for q in ALL_QUERIES:
    qid     = q["query_id"]
    qtext   = q["query_text"]
    ranked  = PER_QUERY[qid].get("stage_b_ranked", [])
    if not ranked:
        PER_QUERY[qid]["stage_c_scored"] = {}
        continue
    targets       = ALL_TARGETS.get(qid, {}) or {}
    t_stat_n      = len(targets.get("statute_targets", []) or [])
    t_conc_n      = len(targets.get("concept_targets_en", []) or [])
    aspects       = ALL_HYDE_ASPECTS.get(qid, []) or []
    aspect_labels = []
    for a in aspects:
        atxt = (a.get("paragraph") or a.get("answer") or "") if isinstance(a, dict) else str(a)
        aspect_labels.append(atxt[:200])

    aspect_legend = ""
    if aspect_labels:
        aspect_legend = "QUERY ASPECTS (the question's sub-issues):\n" + "\n".join(
            f"  #{i+1}: {a[:200]}" for i, a in enumerate(aspect_labels)
        )

    dids = [d for d, _ in ranked]
      # Build all batch prompts upfront, then dispatch as ONE vLLM batched call.
      # Pre-checks each prompt's token count and adaptively splits batches that would
      # exceed the context window (vLLM raises VLLMValidationError instead of
      # gracefully truncating; the recursive split-on-failure can't fix this because
      # bisecting doesn't make any single prompt shorter).

      # Budget: model context − max_new_tokens − safety margin
    PROMPT_TOKEN_BUDGET = MAX_MODEL_LEN - STAGE_C_MAX_NEW_TOKENS - 64

    def _count_tokens(system: str, user: str) -> int:
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        try:
            rendered = llm_tok.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
        except TypeError:
            rendered = llm_tok.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True
            )
        return len(llm_tok(rendered, add_special_tokens=False)["input_ids"])

    def _build_user(sub_batch_dids):
        blocks = []
        for j, did in enumerate(sub_batch_dids, start=1):
            d    = PER_QUERY[qid]["dossier"].get(did, {})
            text = (search_text.get(did, "") or "")[:400].replace("\n", " ")
            blocks.append(_c_format(j, did, d, text, t_stat_n, t_conc_n, aspect_labels))
        return (
            f"QUERY: {qtext}\n\n{aspect_legend}\n\n"
            f"CANDIDATES (N={len(sub_batch_dids)}):{''.join(blocks)}\n"
            f"Output strict JSON array. One object per candidate. No preamble."
        )

    def _emit_safely(candidates, batches_out, msg_pairs_out, depth=0):
        """Try to emit one batch; if too long, halve it. Single-candidate
        prompts that still overflow get emitted with truncated text (fallback)."""
        if not candidates:
            return
        user = _build_user(candidates)
        n_tok = _count_tokens(S2_SYSTEM, user)
        if n_tok <= PROMPT_TOKEN_BUDGET:
            batches_out.append(candidates)
            msg_pairs_out.append((S2_SYSTEM, user))
            return
        # Single candidate that still overflows: trim its text excerpt aggressively.
        if len(candidates) == 1:
            # Build a shrunken version by halving the 400-char text window. At worst
            # this emits a stub-only block; rare enough we just take it.
            did = candidates[0]
            d   = PER_QUERY[qid]["dossier"].get(did, {})
            short_text = (search_text.get(did, "") or "")[:120].replace("\n", " ")
            stub_block = _c_format(1, did, d, short_text, t_stat_n, t_conc_n, aspect_labels)
            short_user = (
                f"QUERY: {qtext}\n\n{aspect_legend}\n\n"
                f"CANDIDATES (N=1):{stub_block}\n"
                f"Output strict JSON array. One object per candidate. No preamble."
            )
            n_tok2 = _count_tokens(S2_SYSTEM, short_user)
            if n_tok2 > PROMPT_TOKEN_BUDGET and depth == 0:
                # Even the stub doesn't fit — typically means aspect_legend or
                # dossier itself is too big. Caller falls back to default score=1.
                print(f"    [Stage C] {qid}: skipping {did} (cannot fit in ctx even trimmed)")
                return
            batches_out.append([did])
            msg_pairs_out.append((S2_SYSTEM, short_user))
            return
        # Split in half and try each side.
        mid = len(candidates) // 2
        _emit_safely(candidates[:mid], batches_out, msg_pairs_out, depth + 1)
        _emit_safely(candidates[mid:], batches_out, msg_pairs_out, depth + 1)

    batches    = []
    msg_pairs  = []
    _oversize_splits = 0
    for b0 in range(0, len(dids), STAGE_C_BATCH):
        batch = dids[b0:b0 + STAGE_C_BATCH]
        _n_before = len(batches)
        _emit_safely(batch, batches, msg_pairs)
        if len(batches) - _n_before > 1:
            _oversize_splits += 1
    if _oversize_splits:
        print(f"    [Stage C] {qid}: {_oversize_splits} oversized batch(es) auto-split to fit ctx")

    _t_q = time.time()
    resps = llm_generate_batch(msg_pairs, max_new_tokens=STAGE_C_MAX_NEW_TOKENS)
    scored = {}
    for batch, resp in zip(batches, resps):
        parsed = _c_parse(resp, len(batch))
        for i, did in enumerate(batch, start=1):
            v = parsed.get(i, {"score": 1, "addresses_aspect": 0, "why": ""})
            scored[did] = v
    n_batches = len(batches)

    PER_QUERY[qid]["stage_c_scored"] = scored

    _g_set = ALL_GOLD_DOC_SET.get(qid, set())
    keep   = [d for d, v in scored.items() if v["score"] >= STAGE_C_KEEP_THRESHOLD]
    gold_kept = len(set(keep) & _g_set)
    score_hist = Counter(v["score"] for v in scored.values())
    print(f"  [{qid}] {time.time()-_t_q:6.1f}s  {n_batches:>3} batches  "
          f"hist={dict(score_hist)}  keep≥{STAGE_C_KEEP_THRESHOLD}: {len(keep):>3}  "
          f"gold_in_keep: {gold_kept}/{len(_g_set)}")
print("[Stage C] complete.")


[Stage C] listwise scoring on Stage-B top-2000 per query (batch=12)...
    [Stage C] val_001: 166 oversized batch(es) auto-split to fit ctx
  [val_001]  209.6s  333 batches  hist={1: 604, 2: 648, 4: 383, 3: 358, 5: 7}  keep≥3: 748  gold_in_keep: 9/42
    [Stage C] val_002: 166 oversized batch(es) auto-split to fit ctx
  [val_002]  200.4s  333 batches  hist={1: 684, 2: 551, 4: 262, 3: 495, 5: 8}  keep≥3: 765  gold_in_keep: 4/38
    [Stage C] val_003: 166 oversized batch(es) auto-split to fit ctx


## Stage D — Pointwise evidence-quoted verdict (Qwen3-32B)

The anti-hallucination gate. For each Stage-C survivor, the LLM must produce a VERBATIM substring of the candidate text as `evidence_quote`. We *verify* the substring with regex against `search_text[did]`. If the quote isn't a substring, verdict is forced to DROP regardless of confidence.

Verdict computed deterministically:
- confidence ≥ 4 + verbatim quote → KEEP
- confidence == 3 + verbatim quote → MAYBE
- confidence ≤ 2 OR empty/invalid quote → DROP


In [ ]:
# Stage D — Pointwise judgment with evidence-quote (anti-hallucination gate)
# For each survivor of Stage C (score ≥ 3), the LLM produces:
#   topic — what this doc is about (1 sentence)
#   relation — how it relates to the query (≤ 2 sentences)
#   evidence_quote — VERBATIM substring of the doc text; MUST be a substring
#   confidence — 1-5
#   addresses_aspect — int, 1..n_aspects
#   verdict — deterministically computed from confidence + non-empty verbatim quote
#
# Batch = 5 candidates per call. Full 1500-char text shown.

import json as _json_d

STAGE_D_BATCH = 5
STAGE_D_FROM_C_MIN_SCORE = 3       # all Stage-C scored >= 3 go through D
STAGE_D_MAX_CANDIDATES   = 60      # cap pointwise to top-60 per query (was unbounded — Stage C left 200-400 → 60 min/query)
STAGE_D_MAX_NEW_TOKENS   = 1500    # was 2500 — output is ~5 items × ~250 tokens (with evidence) = 1250, 1500 has headroom

S3_SYSTEM = (
    "You are a Swiss legal citation expert. For each candidate Swiss legal document, "
    "decide whether it is a relevant citation for the English question.\n\n"
    "CRITICAL ANTI-HALLUCINATION RULES — non-negotiable:\n"
    "1. Judge ONLY from the document text shown. Do NOT invent citations, facts, or rules.\n"
    "2. `evidence_quote` MUST be a VERBATIM substring of the document text. If you cannot "
    "quote a specific sentence proving relevance, set `evidence_quote = \"\"` and "
    "verdict will be DROP.\n"
    "3. Lexical overlap with query terms is NOT evidence. Only legal reasoning that "
    "addresses the question is evidence.\n"
    "4. Procedural-history, facts, dispositif, and notification paragraphs are not "
    "gold even if they mention the right statute.\n\n"
    "DOMAIN FACTS:\n"
    "- Documents are DE / FR / IT. Code aliases (same code): CC=ZGB, CO=OR, CP=StGB, "
    "CPP=StPO, LP=SchKG, LIFD=DBG, LPGA=ATSG, LAI=IVG, LTF=BGG, LAA=UVG, Cst=BV.\n"
    "- The dossier [signals] block per candidate gives PRE-COMPUTED evidence (statute "
    "matches, chamber match, doctrinal density). Use the dossier to guide attention, "
    "verify against the full text.\n\n"
    "VERDICT (computed automatically, but provide consistent values):\n"
    "  confidence 5 + non-empty evidence_quote → KEEP\n"
    "  confidence 4 + non-empty evidence_quote → KEEP\n"
    "  confidence 3 + non-empty evidence_quote → MAYBE\n"
    "  confidence ≤ 2 or empty evidence_quote → DROP\n\n"
    "OUTPUT: strict JSON array. Schema:\n"
    '[{"id": <int>, "topic": "<1 sentence>", "relation": "<≤ 2 sentences>", '
    '"evidence_quote": "<verbatim substring or empty>", "confidence": <1-5>, '
    '"addresses_aspect": <int>, "verdict": "<KEEP|MAYBE|DROP>"}, ...]'
)

def _d_fmt(idx, did, d, text_1500, t_stat_n, t_conc_n, aspect_labels):
    cit  = doc_meta.get(did, {}).get("citation") or did
    role = d["paragraph_role"] or "n/a"
    ch_part = (f"chamber {d['chamber']} ({d['chamber_class']})"
               if d["chamber"] else "(no chamber)")
    ch_tag  = " ← matches query area" if d["chamber_match"] else ""
    asp_str = aspect_labels[d["best_aspect"]][:60] if aspect_labels and 0 <= d["best_aspect"] < len(aspect_labels) else "?"
    full_only = [s for s in d["stat_overlap_full"] if s not in d["stat_overlap_lead"]]
    return (
        f"\n[{idx}] {cit}  ({d['language']}, {d['family']}, {ch_part}{ch_tag}, role={role})\n"
        f"  cites in lead: {{{', '.join(d['stat_overlap_lead']) or '(none)'}}}  "
        f"+ rest: {{{', '.join(full_only) or '(none)'}}}\n"
        f"  concepts caught: {{{', '.join(d['conc_overlap']) or '(none)'}}}\n"
        f"  best aspect: #{d['best_aspect']+1} \"{asp_str}…\"  pool_cited_by={d['co_cite_count']}\n"
        f"  doctrinal_density={d['doctrinal']:.2f}\n"
        f"  full_text: \"{text_1500}\"\n"
    )

def _d_parse(resp, n_expected, text_lookup):
    m = re.search(r"\[.*\]", resp, re.S)
    if not m: return {}
    try: arr = _json_d.loads(m.group(0))
    except Exception: return {}
    if not isinstance(arr, list): return {}
    out = {}
    for item in arr:
        if not isinstance(item, dict): continue
        try:
            idx = int(item.get("id", -1))
            if not (1 <= idx <= n_expected): continue
            conf  = max(1, min(5, int(item.get("confidence", 0))))
            ev    = str(item.get("evidence_quote", ""))[:600]
            topic = str(item.get("topic", ""))[:300]
            rel   = str(item.get("relation", ""))[:500]
            asp   = max(0, int(item.get("addresses_aspect", 1)) - 1)

            # Verify quote is verbatim substring (whitespace-normalized)
            ev_clean = re.sub(r"\s+", " ", ev).strip()
            text_clean = re.sub(r"\s+", " ", text_lookup.get(idx, ""))
            quote_valid = bool(ev_clean) and (ev_clean in text_clean)

            # Deterministic verdict
            if not quote_valid:        verdict = "DROP"
            elif conf <= 2:            verdict = "DROP"
            elif conf == 3:            verdict = "MAYBE"
            else:                      verdict = "KEEP"

            out[idx] = {
                "topic":             topic,
                "relation":          rel,
                "evidence_quote":    ev,
                "evidence_verbatim": quote_valid,
                "confidence":        conf,
                "addresses_aspect":  asp,
                "verdict":           verdict,
            }
        except (TypeError, ValueError):
            continue
    return out

print(f"[Stage D] pointwise judgment on Stage-C survivors (batch={STAGE_D_BATCH})...")
for q in ALL_QUERIES:
    qid           = q["query_id"]
    qtext         = q["query_text"]
    scored_c      = PER_QUERY[qid].get("stage_c_scored", {})
    survivors     = [d for d, v in scored_c.items() if v["score"] >= STAGE_D_FROM_C_MIN_SCORE]
    # Sort by (Stage-C score, then Stage-B composite score) and cap at STAGE_D_MAX_CANDIDATES
    score_b_map   = PER_QUERY[qid].get("stage_b_score_map", {})
    survivors.sort(key=lambda d: (-scored_c[d]["score"], -score_b_map.get(d, 0.0)))
    survivors     = survivors[:STAGE_D_MAX_CANDIDATES]
    if not survivors:
        PER_QUERY[qid]["stage_d_verdicts"] = {}
        continue
    targets       = ALL_TARGETS.get(qid, {}) or {}
    t_stat_n      = len(targets.get("statute_targets", []) or [])
    t_conc_n      = len(targets.get("concept_targets_en", []) or [])
    aspects       = ALL_HYDE_ASPECTS.get(qid, []) or []
    aspect_labels = []
    for a in aspects:
        atxt = (a.get("paragraph") or a.get("answer") or "") if isinstance(a, dict) else str(a)
        aspect_labels.append(atxt[:200])

    aspect_legend = ""
    if aspect_labels:
        aspect_legend = "QUERY ASPECTS (the question's sub-issues):\n" + "\n".join(
            f"  #{i+1}: {a[:200]}" for i, a in enumerate(aspect_labels)
        )

    # Build all batch prompts upfront, dispatch as ONE batched vLLM call.
    batches      = []
    text_lookups = []
    msg_pairs    = []
    for b0 in range(0, len(survivors), STAGE_D_BATCH):
        batch = survivors[b0:b0 + STAGE_D_BATCH]
        blocks = []
        text_lookup = {}
        for i, did in enumerate(batch, start=1):
            d    = PER_QUERY[qid]["dossier"].get(did, {})
            text = (search_text.get(did, "") or "")[:1500].replace("\n", " ")
            text_lookup[i] = text
            blocks.append(_d_fmt(i, did, d, text, t_stat_n, t_conc_n, aspect_labels))
        user = (
            f"QUERY: {qtext}\n\n{aspect_legend}\n\n"
            f"CANDIDATES (N={len(batch)}):{''.join(blocks)}\n"
            f"For each candidate, output strict JSON. evidence_quote MUST be verbatim "
            f"substring of full_text; if you can't quote, set it to empty string. "
            f"No preamble, just the JSON array."
        )
        batches.append(batch)
        text_lookups.append(text_lookup)
        msg_pairs.append((S3_SYSTEM, user))

    _t_q = time.time()
    resps = llm_generate_batch(msg_pairs, max_new_tokens=STAGE_D_MAX_NEW_TOKENS)
    verdicts = {}
    for batch, text_lookup, resp in zip(batches, text_lookups, resps):
        parsed = _d_parse(resp, len(batch), text_lookup)
        for i, did in enumerate(batch, start=1):
            v = parsed.get(i, {
                "topic": "", "relation": "", "evidence_quote": "",
                "evidence_verbatim": False, "confidence": 0,
                "addresses_aspect": 0, "verdict": "DROP",
            })
            verdicts[did] = v
    n_batches = len(batches)

    PER_QUERY[qid]["stage_d_verdicts"] = verdicts

    keep  = [d for d, v in verdicts.items() if v["verdict"] == "KEEP"]
    maybe = [d for d, v in verdicts.items() if v["verdict"] == "MAYBE"]
    drop  = [d for d, v in verdicts.items() if v["verdict"] == "DROP"]
    _g_set    = ALL_GOLD_DOC_SET.get(qid, set())
    g_keep    = len(set(keep) & _g_set)
    g_maybe   = len(set(maybe) & _g_set)
    print(f"  [{qid}] {time.time()-_t_q:6.1f}s  {n_batches:>3} batches  "
          f"KEEP={len(keep):>3} (gold {g_keep:>2}/{len(_g_set)})  "
          f"MAYBE={len(maybe):>3} (gold {g_maybe:>2})  DROP={len(drop):>3}")
print("[Stage D] complete.")


# Phase 4 — Final selection


## Stage E — Adaptive K + aspect coverage + case dedup (no LLM)

Variable-size final selection. No fixed K.

1. Start with all KEEP@5
2. Add all KEEP@4
3. For each query aspect with no KEEP: promote the highest-confidence MAYBE addressing that aspect
4. Case-level dedup: per `court_base`, keep best (confidence then composite score); never two same-case paragraphs unless they address different aspects.


In [ ]:
# Stage E — Adaptive K + aspect coverage + case-level dedup (no LLM)
# Variable-size final selection driven by confidence + aspect coverage.
# - Start: all KEEP@5
# - Add: all KEEP@4
# - Aspect-gap fill: for each aspect with no KEEP, promote highest-confidence MAYBE on that aspect
# - Case dedup: per case_base, keep highest-confidence; never two same-case paragraphs unless
#   they address different aspects.
# NO TOP-K CAP. Output size = whatever the LLM conviction supports.

print("[Stage E] adaptive selection + aspect coverage + case dedup...")
for q in ALL_QUERIES:
    qid       = q["query_id"]
    verdicts  = PER_QUERY[qid].get("stage_d_verdicts", {})
    aspects   = ALL_HYDE_ASPECTS.get(qid, []) or []
    n_aspects = max(1, len(aspects))

    keeps_5  = [d for d, v in verdicts.items() if v["verdict"] == "KEEP" and v["confidence"] == 5]
    keeps_4  = [d for d, v in verdicts.items() if v["verdict"] == "KEEP" and v["confidence"] == 4]
    maybes   = [d for d, v in verdicts.items() if v["verdict"] == "MAYBE"]

    # Sort each tier by stage_b composite score (deterministic ordering for ties)
    score_b = PER_QUERY[qid].get("stage_b_score_map", {})
    sort_key = lambda d: -score_b.get(d, 0.0)
    keeps_5.sort(key=sort_key)
    keeps_4.sort(key=sort_key)
    maybes.sort(key=sort_key)

    # Initial selection
    selected = list(keeps_5) + list(keeps_4)
    promoted_log = {}

    # Aspect-gap fill
    covered_aspects = {verdicts[d]["addresses_aspect"] for d in selected}
    for aspect_idx in range(n_aspects):
        if aspect_idx in covered_aspects: continue
        # Find best MAYBE addressing this aspect
        cands = [d for d in maybes
                 if verdicts[d]["addresses_aspect"] == aspect_idx
                 and d not in selected]
        if cands:
            best = max(cands, key=lambda d: (verdicts[d]["confidence"], score_b.get(d, 0.0)))
            selected.append(best)
            promoted_log[best] = f"aspect#{aspect_idx+1}"

    # Case-level dedup: per case_base, keep best by (confidence, then composite score).
    # EXCEPT keep multiple if they address DIFFERENT aspects (rare).
    by_case = defaultdict(list)
    for did in selected:
        cb = (doc_meta.get(did, {}).get("court_base") or "").strip()
        if not cb or doc_meta.get(did, {}).get("family") == "law":
            by_case[("__solo__", did)].append(did)  # laws are kept individually
        else:
            by_case[cb].append(did)

    dedup = []
    dedup_dropped = []
    for cb, dids in by_case.items():
        if cb[0] == "__solo__":
            dedup.extend(dids); continue
        if len(dids) == 1:
            dedup.append(dids[0]); continue
        # Group by aspect within this case
        by_asp = defaultdict(list)
        for d in dids:
            by_asp[verdicts[d]["addresses_aspect"]].append(d)
        for asp_idx, asp_dids in by_asp.items():
            best = max(asp_dids, key=lambda d: (verdicts[d]["confidence"], score_b.get(d, 0.0)))
            dedup.append(best)
            for other in asp_dids:
                if other != best:
                    dedup_dropped.append(other)

    PER_QUERY[qid]["stage_e_selected"] = dedup
    PER_QUERY[qid]["stage_e_promoted"] = promoted_log
    PER_QUERY[qid]["stage_e_case_dropped"] = dedup_dropped

    _g_set       = ALL_GOLD_DOC_SET.get(qid, set())
    n_K          = len(dedup)
    gold_caught  = len(set(dedup) & _g_set)
    gold_total   = len(_g_set)
    precision    = gold_caught / max(1, n_K)
    recall       = gold_caught / max(1, gold_total)
    f1           = (2*precision*recall / (precision+recall)) if (precision+recall) else 0.0
    print(f"  [{qid}] K={n_K:>3}  gold {gold_caught}/{gold_total}  "
          f"P={precision:.3f} R={recall:.3f} F1={f1:.3f}  "
          f"(K5={len(keeps_5)} K4={len(keeps_4)} M={len(maybes)} "
          f"promoted={len(promoted_log)} case_dedup={len(dedup_dropped)})")

print("[Stage E] complete.")


## Stage F — Self-validation pass (Qwen3-32B, 1 call per query)

Precision booster. Present the Stage-E selection as a complete set; ask Qwen3-32B "is each item necessary or redundant given the rest?". Drop the redundant; **safety net**: never drop the only candidate for an aspect (if it would, rescue it).


In [ ]:
# Stage F — Self-validation pass (precision booster)
# For each query, present the Stage-E selection to Qwen3-32B as a SET and ask:
# "Is each item necessary or redundant given the others?"
# Drop the items the LLM flags as redundant. Then sanity check aspect coverage stays intact.

import json as _json_f

SF_SYSTEM = (
    "You are a Swiss legal citation expert reviewing a final set of citations a system "
    "proposes to deliver as the answer to a legal question. Your job is to identify "
    "any item that is REDUNDANT given the rest of the set — i.e., another item already "
    "covers the same legal point with equal or greater authority.\n\n"
    "RULES:\n"
    "1. A leading case (BGE published) generally trumps a parallel unpublished decision "
    "on the same legal point.\n"
    "2. A statute (law article) trumps a case that merely cites that statute without "
    "applying new doctrine, IF both are in the set.\n"
    "3. Two paragraphs of the same case are redundant UNLESS they address different "
    "legal sub-issues.\n"
    "4. Keep all unique-aspect coverage — do not drop something that is the only "
    "candidate for a sub-issue.\n"
    "5. When in doubt, KEEP. Precision matters but missing the only answer to an aspect "
    "is worse than keeping one redundant item.\n\n"
    "OUTPUT: strict JSON array, one object per item. Schema:\n"
    '[{"id": <int>, "necessary": true|false, "reason": "<≤15 words>"}, ...]\n'
    "No preamble."
)

def _f_fmt(idx, did, d, verdict, text_400):
    cit  = doc_meta.get(did, {}).get("citation") or did
    role = d["paragraph_role"] or "n/a"
    return (
        f"\n[{idx}] {cit}  ({d['language']}, {d['family']}, role={role})\n"
        f"  topic: {verdict['topic']}\n"
        f"  relation: {verdict['relation']}\n"
        f"  evidence: \"{verdict['evidence_quote'][:200]}\"\n"
        f"  addresses aspect #{verdict['addresses_aspect']+1}; confidence={verdict['confidence']}\n"
        f"  text excerpt: {text_400}\n"
    )

def _f_parse(resp, n_expected):
    m = re.search(r"\[.*\]", resp, re.S)
    if not m: return {}
    try: arr = _json_f.loads(m.group(0))
    except Exception: return {}
    if not isinstance(arr, list): return {}
    out = {}
    for item in arr:
        if not isinstance(item, dict): continue
        try:
            idx = int(item.get("id", -1))
            if not (1 <= idx <= n_expected): continue
            nec = bool(item.get("necessary", True))
            rsn = str(item.get("reason", ""))[:200]
            out[idx] = {"necessary": nec, "reason": rsn}
        except (TypeError, ValueError):
            continue
    return out

print("[Stage F] self-validation (1 LLM call per query)...")
for q in ALL_QUERIES:
    qid       = q["query_id"]
    selected  = PER_QUERY[qid].get("stage_e_selected", [])
    verdicts  = PER_QUERY[qid].get("stage_d_verdicts", {})
    dossier   = PER_QUERY[qid].get("dossier", {})
    if not selected:
        PER_QUERY[qid]["stage_f_final"] = []
        PER_QUERY[qid]["stage_f_dropped"] = []
        continue

    blocks = []
    for i, did in enumerate(selected, start=1):
        d    = dossier.get(did, {})
        v    = verdicts.get(did, {})
        text = (search_text.get(did, "") or "")[:400].replace("\n", " ")
        blocks.append(_f_fmt(i, did, d, v, text))

    aspects = ALL_HYDE_ASPECTS.get(qid, []) or []
    aspect_labels = []
    for a in aspects:
        atxt = (a.get("paragraph") or a.get("answer") or "") if isinstance(a, dict) else str(a)
        aspect_labels.append(atxt[:200])

    aspect_legend = ""
    if aspect_labels:
        aspect_legend = "QUERY ASPECTS:\n" + "\n".join(
            f"  #{i+1}: {a[:200]}" for i, a in enumerate(aspect_labels)
        )

    user = (
        f"QUERY: {q['query_text']}\n\n{aspect_legend}\n\n"
        f"PROPOSED FINAL SET (N={len(selected)}):{''.join(blocks)}\n\n"
        f"For each item, output strict JSON {{id, necessary, reason}}. "
        f"Only mark `necessary=false` if another item in this set already covers the same "
        f"legal point AND with equal/greater authority. Output ONLY the JSON array."
    )

    _t = time.time()
    resp   = llm_generate(SF_SYSTEM, user, max_new_tokens=800)  # was 1500 — output is ~K × 25 tokens, 800 has headroom
    parsed = _f_parse(resp, len(selected))

    kept    = []
    dropped = []
    for i, did in enumerate(selected, start=1):
        result = parsed.get(i, {"necessary": True, "reason": "(no judgment, keeping)"})
        if result["necessary"]:
            kept.append(did)
        else:
            dropped.append((did, result["reason"]))

    # Safety: never drop the only candidate for an aspect.
    covered = {verdicts[d]["addresses_aspect"] for d in kept}
    rescued = []
    for did, reason in dropped:
        asp = verdicts.get(did, {}).get("addresses_aspect", 0)
        if asp not in covered:
            kept.append(did)
            rescued.append((did, asp))
            covered.add(asp)
    dropped = [(d, r) for d, r in dropped if not any(rd == d for rd, _ in rescued)]

    PER_QUERY[qid]["stage_f_final"]   = kept
    PER_QUERY[qid]["stage_f_dropped"] = dropped

    _g_set    = ALL_GOLD_DOC_SET.get(qid, set())
    n_K       = len(kept)
    gold_caught = len(set(kept) & _g_set)
    gold_total  = len(_g_set)
    precision = gold_caught / max(1, n_K)
    recall    = gold_caught / max(1, gold_total)
    f1        = (2*precision*recall / (precision+recall)) if (precision+recall) else 0.0
    print(f"  [{qid}] {time.time()-_t:5.1f}s  "
          f"{len(selected)} → {n_K}  (dropped {len(dropped)}, rescued {len(rescued)})  "
          f"gold {gold_caught}/{gold_total}  P={precision:.3f} R={recall:.3f} F1={f1:.3f}")
print("[Stage F] complete.")


# Phase 5 — Eval + save


In [ ]:
# Phase 5 — Evaluation + save
# Compute per-query and aggregate P / R / F1 vs gold. Save final selection +
# full pipeline state to disk for inspection / leaderboard submission.

import json as _json_e

# ───────── 1) Per-query metrics table ─────────
print()
print("=" * 102)
print("  PRECISION-V1 — FINAL RESULTS (variable K, evidence-quote-gated)")
print("=" * 102)
print(f"  {'query':<10}{'gold':>5}{'K':>5}{'TP':>5}"
      f"{'P':>8}{'R':>8}{'F1':>8}{'aspects_cov':>13}  selection_breakdown")
print("-" * 102)

_macro_p, _macro_r, _macro_f1 = [], [], []
_tot_tp, _tot_pred, _tot_gold = 0, 0, 0

for q in ALL_QUERIES:
    qid     = q["query_id"]
    final   = PER_QUERY[qid].get("stage_f_final", [])
    _g_set  = ALL_GOLD_DOC_SET.get(qid, set())
    n_K     = len(final)
    n_TP    = len(set(final) & _g_set)
    n_gold  = len(_g_set)
    P       = n_TP / max(1, n_K)
    R       = n_TP / max(1, n_gold)
    F1      = (2*P*R / (P+R)) if (P+R) else 0.0

    # Aspect coverage breakdown
    verdicts  = PER_QUERY[qid].get("stage_d_verdicts", {})
    aspects   = ALL_HYDE_ASPECTS.get(qid, []) or []
    n_asp     = max(1, len(aspects))
    covered   = {verdicts.get(d, {}).get("addresses_aspect", 0) for d in final}
    asp_str   = f"{len(covered & set(range(n_asp)))}/{n_asp}"

    # Breakdown: how many K5/K4/MAYBE/promoted/dedup_dropped contributed?
    keeps_5 = sum(1 for d in final if verdicts.get(d, {}).get("confidence") == 5 and verdicts.get(d, {}).get("verdict") == "KEEP")
    keeps_4 = sum(1 for d in final if verdicts.get(d, {}).get("confidence") == 4 and verdicts.get(d, {}).get("verdict") == "KEEP")
    maybe_n = sum(1 for d in final if verdicts.get(d, {}).get("verdict") == "MAYBE")
    breakdown = f"K5={keeps_5} K4={keeps_4} M={maybe_n}"

    print(f"  {qid:<10}{n_gold:>5}{n_K:>5}{n_TP:>5}{P:>8.3f}{R:>8.3f}{F1:>8.3f}{asp_str:>13}  {breakdown}")

    _macro_p.append(P);   _macro_r.append(R);   _macro_f1.append(F1)
    _tot_tp += n_TP;      _tot_pred += n_K;     _tot_gold += n_gold

print("-" * 102)
_macP  = sum(_macro_p) / len(_macro_p)
_macR  = sum(_macro_r) / len(_macro_r)
_macF1 = sum(_macro_f1) / len(_macro_f1)
_miP   = _tot_tp / max(1, _tot_pred)
_miR   = _tot_tp / max(1, _tot_gold)
_miF1  = (2*_miP*_miR / (_miP+_miR)) if (_miP+_miR) else 0.0
print(f"  {'MACRO':<10}{'':>5}{'':>5}{'':>5}{_macP:>8.3f}{_macR:>8.3f}{_macF1:>8.3f}")
print(f"  {'MICRO':<10}{_tot_gold:>5}{_tot_pred:>5}{_tot_tp:>5}{_miP:>8.3f}{_miR:>8.3f}{_miF1:>8.3f}")

# ───────── 2) Comparison vs fusion baseline at same K per query ─────────
print()
print("=" * 102)
print("  CASCADE vs FUSION-BASELINE F1 (at the same K per query)")
print("=" * 102)
print(f"  {'query':<10}{'K':>5}{'fusion F1':>12}{'cascade F1':>13}{'Δ':>+8}")
_baseline_f1, _cascade_better = [], 0
for q in ALL_QUERIES:
    qid     = q["query_id"]
    final   = PER_QUERY[qid].get("stage_f_final", [])
    _g_set  = ALL_GOLD_DOC_SET.get(qid, set())
    n_K     = len(final)
    n_gold  = len(_g_set)
    F1_c    = _macro_f1[ALL_QUERIES.index(q)]
    # Fusion R@n_K from curve
    curve   = PER_QUERY[qid].get("curve", {})
    keys    = sorted(curve.keys())
    kk      = max((k for k in keys if k <= n_K), default=None)
    fusion_R= curve[kk][1] if kk is not None else 0.0
    fusion_TP = int(round(fusion_R * n_gold))
    fusion_P  = fusion_TP / max(1, n_K)
    F1_f    = (2*fusion_P*fusion_R / (fusion_P+fusion_R)) if (fusion_P+fusion_R) else 0.0
    _baseline_f1.append(F1_f)
    if F1_c > F1_f: _cascade_better += 1
    print(f"  {qid:<10}{n_K:>5}{F1_f:>12.3f}{F1_c:>13.3f}{F1_c-F1_f:>+8.3f}")
print("-" * 102)
print(f"  cascade wins on {_cascade_better}/{len(ALL_QUERIES)} queries  "
      f"(macro fusion F1 = {sum(_baseline_f1)/len(_baseline_f1):.3f} vs "
      f"macro cascade F1 = {_macF1:.3f})")

# ───────── 3) Save results ─────────
summary = {
    "macro_P":  _macP, "macro_R":  _macR, "macro_F1":  _macF1,
    "micro_P":  _miP,  "micro_R":  _miR,  "micro_F1":  _miF1,
    "n_queries": len(ALL_QUERIES),
    "cascade_wins": _cascade_better,
    "fusion_baseline_macro_F1": sum(_baseline_f1) / len(_baseline_f1),
    "per_query": {
        q["query_id"]: {
            "query_text": q["query_text"],
            "gold":       len(ALL_GOLD_DOC_SET.get(q["query_id"], set())),
            "K":          len(PER_QUERY[q["query_id"]].get("stage_f_final", [])),
            "TP":         len(set(PER_QUERY[q["query_id"]].get("stage_f_final", [])) & ALL_GOLD_DOC_SET.get(q["query_id"], set())),
            "P":          _macro_p[i],
            "R":          _macro_r[i],
            "F1":         _macro_f1[i],
            "selected":   PER_QUERY[q["query_id"]].get("stage_f_final", []),
            "verdicts":   {d: PER_QUERY[q["query_id"]]["stage_d_verdicts"].get(d, {})
                           for d in PER_QUERY[q["query_id"]].get("stage_f_final", [])},
            "stage_e_promoted":    PER_QUERY[q["query_id"]].get("stage_e_promoted", {}),
            "stage_f_dropped":     [d for d, _ in PER_QUERY[q["query_id"]].get("stage_f_dropped", [])],
            "stage_b_R_top500":    None,  # filled below
        }
        for i, q in enumerate(ALL_QUERIES)
    },
}
# Add Stage-B recall snapshot too
for q in ALL_QUERIES:
    qid    = q["query_id"]
    top_b  = {d for d, _ in PER_QUERY[qid].get("stage_b_ranked", [])}
    _g     = ALL_GOLD_DOC_SET.get(qid, set())
    summary["per_query"][qid]["stage_b_R_top500"] = len(top_b & _g) / max(1, len(_g))

(OUT_DIR / "precision_v1_summary.json").write_text(
    _json_e.dumps(summary, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
print(f"\n[save] wrote {OUT_DIR / 'precision_v1_summary.json'}")

# Per-query citation strings for Kaggle-style submission
sub = {}
for q in ALL_QUERIES:
    qid    = q["query_id"]
    finals = PER_QUERY[qid].get("stage_f_final", [])
    cits   = []
    for did in finals:
        cit = doc_meta.get(did, {}).get("citation", did) or did
        cits.append(cit)
    sub[qid] = cits
(OUT_DIR / "precision_v1_predictions.json").write_text(
    _json_e.dumps(sub, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"[save] wrote {OUT_DIR / 'precision_v1_predictions.json'} (citation strings per query)")
